# Phase 1: Dataset Collection & Schema Inspection

**Project:** Indian E-Commerce Recommendation System  
**Objective:** Discover, inspect, and document the 100k Amazon Products dataset across all 9 category CSV files.

### Dataset Overview
- Source: Kaggle (*A Comprehensive Dataset of 100k Amazon Products*)
- Target Directory: `data/raw/`
- Focus: Identify actual columns, data types, encoding issues, missing values, and formatting quirks before writing preprocessing logic.

In [ ]:
import os
import pandas as pd
import numpy as np

# Relative path from notebooks/ to data/raw
RAW_DIR = os.path.join("..", "data", "raw")
files = sorted([f for f in os.listdir(RAW_DIR) if f.endswith('.csv')])
print(f"Total raw CSV files found: {len(files)}")
for idx, f in enumerate(files, 1):
    print(f"{idx}. {f}")

## 1. Inspecting Encoding, File Sizes, and Line Counts
We read each file line-by-line to check physical record counts, then load sample rows using `latin1` encoding.

In [ ]:
inspection_results = []

for f in files:
    path = os.path.join(RAW_DIR, f)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    
    with open(path, 'rb') as file_obj:
        line_count = sum(1 for _ in file_obj)
        
    # Load with pandas
    df = pd.read_csv(path, encoding='latin1', on_bad_lines='skip', low_memory=False)
    
    inspection_results.append({
        'file': f,
        'size_mb': round(size_mb, 2),
        'physical_lines': line_count,
        'parsed_rows': len(df),
        'columns_count': len(df.columns),
        'columns': list(df.columns)
    })

summary_df = pd.DataFrame(inspection_results)
summary_df[['file', 'size_mb', 'physical_lines', 'parsed_rows', 'columns_count']]

## 2. Detailed Schema Analysis Across Categories
Examining the exact column names, missing values, and sample data.

In [ ]:
for res in inspection_results:
    f = res['file']
    path = os.path.join(RAW_DIR, f)
    df = pd.read_csv(path, encoding='latin1', on_bad_lines='skip', low_memory=False)
    
    print(f"\n{'=' * 30} {f} {'=' * 30}")
    print(f"Columns: {list(df.columns)}")
    print("\nMissing Values:")
    print(df.isnull().sum())
    print("\nData Types:")
    print(df.dtypes)
    print("\nSample Record:")
    print(df.iloc[0].dropna().to_dict())

## 3. Summary of Findings & Next Steps (Phase 2)

1. **Encoding**: Must use `latin1` (or `cp1252`) because UTF-8 decoding throws byte errors on characters like `0x96` (en-dashes) and `0xae` (registered trademark).
2. **Description Column Variants**: Must handle `Product Description `, `Product Description`, and `Production Description` uniformly.
3. **Price Column Variants**: Must handle `Price(Dollar)` and `PriceDollar)`.
4. **Missing Values**: Real prices are missing in up to 70% of rows; listed price is populated in 98%+ of records.
5. **Artifact Columns**: Files such as `amazon_camra.csv` and `amazon_mobile.csv` have unescaped commas generating `Unnamed: 7` through `Unnamed: 15`. These will be safely stripped during cleaning.